In [1]:
!pip install polars -q

In [2]:
!pip install catboost -q

In [3]:
import gc
import polars as pl
import pandas as pd
import catboost
from catboost import Pool, CatBoostClassifier

In [4]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [5]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1600/train_combined_1600.parquet')

In [6]:
test = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1600/test_combined_1600.parquet')

In [7]:
target = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_target.parquet')

In [8]:
target_columns = [col for col in target.columns if col.startswith("target")]

In [9]:
cat_feature_names = [
    col_name for col_name in train.columns
    if col_name.startswith("cat_feature")
]

In [10]:
X_train = train.drop('customer_id').to_pandas().astype('float32')
y_train = target.drop('customer_id').to_pandas()

for col in cat_feature_names:
    if col in X_train.columns:
        # Заполняем NaN значением -1 
        X_train[col] = X_train[col].fillna(-1).astype(int)

train_pool = Pool(
    X_train, 
    label=y_train.values, 
    cat_features = cat_feature_names)

del train
gc.collect()

0

In [11]:
catboost_model = CatBoostClassifier(
    iterations=3000,
    depth=8,
    learning_rate=0.02,
    random_seed=42,
    loss_function='MultiLogloss', 
    task_type='GPU',
    l2_leaf_reg=7.0,
    random_strength=1.5,
    verbose=100
)

catboost_model.fit(train_pool)

0:	learn: 0.6454123	total: 2.59s	remaining: 2h 9m 17s
100:	learn: 0.0917967	total: 4m 42s	remaining: 2h 15m 12s
200:	learn: 0.0873540	total: 9m 22s	remaining: 2h 10m 35s
300:	learn: 0.0857004	total: 14m 5s	remaining: 2h 6m 22s
400:	learn: 0.0845679	total: 18m 46s	remaining: 2h 1m 40s
500:	learn: 0.0836588	total: 23m 29s	remaining: 1h 57m 9s
600:	learn: 0.0828953	total: 28m 13s	remaining: 1h 52m 40s
700:	learn: 0.0822270	total: 32m 53s	remaining: 1h 47m 53s
800:	learn: 0.0816282	total: 37m 32s	remaining: 1h 43m 3s
900:	learn: 0.0810711	total: 42m 13s	remaining: 1h 38m 22s
1000:	learn: 0.0805680	total: 46m 50s	remaining: 1h 33m 33s
1100:	learn: 0.0800940	total: 51m 29s	remaining: 1h 28m 48s
1200:	learn: 0.0796563	total: 56m 3s	remaining: 1h 23m 58s
1300:	learn: 0.0792658	total: 1h 30s	remaining: 1h 19m 1s
1400:	learn: 0.0789228	total: 1h 4m 48s	remaining: 1h 13m 57s
1500:	learn: 0.0786014	total: 1h 9m 1s	remaining: 1h 8m 56s
1600:	learn: 0.0783049	total: 1h 13m 13s	remaining: 1h 3m 58s
1

CatBoostClassifier(depth=8, iterations=3000, l2_leaf_reg=7.0, learning_rate=0.02, loss_function='MultiLogloss', random_seed=42, random_strength=1.5, task_type='GPU', verbose=100)

In [12]:
X_test = test.drop('customer_id').to_pandas().astype('float32')

for col in cat_feature_names:
    if col in X_train.columns:
        # Заполняем NaN значением -1 
        X_test[col] = X_test[col].fillna(-1).astype(int)

In [13]:
test_pool = Pool(
    data=X_test,
    cat_features = cat_feature_names
)

y_pred_raw = catboost_model.predict(test_pool, prediction_type='RawFormulaVal')
print(f"Форма сырых значений: {y_pred_raw.shape}")

Форма сырых значений: (250000, 41)


In [14]:
predict_schema = [f"predict_{t.split('target_')[1]}" for t in target_columns]

cat_submit = pl.DataFrame(
    y_pred_raw,
    schema=predict_schema
)

In [15]:
test_ids = pl.read_parquet(
    "/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_main_features.parquet",
    columns=["customer_id"]
)

In [16]:
cat_submit = pl.concat(
    [test_ids, cat_submit],
    how="horizontal"
)

cat_submit.write_parquet("cat_features_1600.parquet")